In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('data/Drug Train Dataframe.csv')

In [3]:
df.shape

(161297, 8)

In [ ]:
df.head()

In [5]:
# Count missing values in each column
missing_counts = df.isna().sum()

# Work out the percentage of rows missing in each column
missing_pct = (df.isna().mean() * 100).round(2)

# Put both into one dataframe so it's easy to read
missing_summary = pd.DataFrame({
    'missing_count': missing_counts,   # number of NaN values
    'missing_percent': missing_pct     # share of all rows, in %
})

# Show the table
missing_summary

,missing_count,missing_percent
Unnamed: 0,0,0.00
uniqueID,0,0.00
drugName,0,0.00
condition,899,0.56
review,0,0.00
rating,0,0.00
date,0,0.00
usefulCount,0,0.00


In [6]:
# Blank means empty text or only spaces, which .isna() cannot catch
# .str.strip() removes spaces from both ends, then we test for ''
blank_condition = (df['condition'].str.strip() == '').sum()
blank_review = (df['review'].str.strip() == '').sum()

# Show both counts in one small dataframe
blank_summary = pd.DataFrame({
    'blank_count': [blank_condition, blank_review]
}, index=['condition', 'review'])

blank_summary

,blank_count
condition,0
review,0


In [7]:
# Count missing condition rows in the current dataframe
missing_condition = df['condition'].isna().sum()

# Compare with the README's number
print('Missing condition rows now:', missing_condition)
print('README says: 899')
print('Match:', missing_condition == 899)

Missing condition rows now: 899
README says: 899
Match: True


In [8]:
# Keep only the rows where condition is missing
missing_rows = df[df['condition'].isna()]

# Count how many missing rows each drug has, keep the top 10
# .to_frame() turns the result into a dataframe with a column name we choose
top_drugs = missing_rows['drugName'].value_counts().head(10).to_frame('missing_count')

# Count missing rows at each rating, then sort by rating (1 to 10)
missing_by_rating = missing_rows['rating'].value_counts().sort_index().to_frame('missing_count')

# Show both tables
display(top_drugs)
display(missing_by_rating)

,missing_count
drugName,
Ethinyl estradiol / norethindrone,100
Ethinyl estradiol / norgestimate,84
Ethinyl estradiol / levonorgestrel,79
Drospirenone / ethinyl estradiol,34
Levonorgestrel,26
Medroxyprogesterone,24
Etonogestrel,15
Desogestrel / ethinyl estradiol,11
Amphetamine / dextroamphetamine,11


,missing_count
rating,
1,115
2,52
3,48
4,32
5,54
6,42
7,61
8,99
9,152


In [9]:
# Keep only the rows that DO have a condition
clean_rows = df[df['condition'].notna()]

# For each drug, calculate four things about its condition column
drug_stats = clean_rows.groupby('drugName')['condition'].agg(
    n_reviews='count',        # how many clean reviews the drug has
    n_conditions='nunique',   # how many different conditions it appears under
    # the most common condition for this drug (first item of value_counts)
    top_condition=lambda x: x.value_counts().index[0],
    # share of the drug's reviews taken by that most common condition (0 to 1)
    top_share=lambda x: x.value_counts(normalize=True).iloc[0]
)

# Round the share so it's easier to read
drug_stats['top_share'] = drug_stats['top_share'].round(3)

In [10]:
# Start from the top 10 missing-condition drugs and attach each drug's stats
# .join() matches on the drug name (the index of both tables)
dominance_table = top_drugs.join(drug_stats)

# Show the table
dominance_table

,missing_count,n_reviews,n_conditions,top_condition,top_share
drugName,,,,,
Ethinyl estradiol / norethindrone,100,2750,8,Birth Control,0.850
Ethinyl estradiol / norgestimate,84,2033,6,Birth Control,0.777
Ethinyl estradiol / levonorgestrel,79,1809,7,Birth Control,0.884
Drospirenone / ethinyl estradiol,34,856,5,Birth Control,0.610
Levonorgestrel,26,3631,4,Birth Control,0.586
Medroxyprogesterone,24,971,7,Birth Control,0.517
Etonogestrel,15,3321,2,Birth Control,0.998
Desogestrel / ethinyl estradiol,11,511,4,Birth Control,0.967
Amphetamine / dextroamphetamine,11,408,4,ADHD,0.865


In [11]:
# Part A: split the data into two tables

# True for rows where condition is missing, False for rows where it is filled in
mask = df['condition'].isna()

# Rows with a missing condition go into their own table
# .copy() makes it a separate table, not a window into df
missing_cond_df = df[mask].copy()

# Rows with a condition go into main_df
# ~ flips True and False, so ~mask means "condition is NOT missing"
main_df = df[~mask].copy()

# Start the row numbers at 0 again in main_df
# drop=True stops pandas from adding the old numbers as a new column
main_df = main_df.reset_index(drop=True)

In [12]:
# Part B: check that nothing got lost

# Print how many rows each table has
print('Original df:     ', len(df))
print('main_df:         ', len(main_df))
print('missing_cond_df: ', len(missing_cond_df))

# assert stops with an error if the statement is False
# If everything adds up, it prints nothing
assert len(main_df) + len(missing_cond_df) == len(df)

Original df:      161297
main_df:          160398
missing_cond_df:  899


In [13]:
# Part C: save the set-aside rows so they survive between sessions

# index=False stops pandas from writing an extra column of row numbers
missing_cond_df.to_csv('data/missing_condition_rows.csv', index=False)

In [14]:
# Cell 1: BEFORE check

# Count how many rows have the misspelled value exactly
# == 'Bipolar Disorde' gives True/False for each row, .sum() counts the Trues
print('Misspelled rows:', (main_df['condition'] == 'Bipolar Disorde').sum())

# Count how many rows already have the correct spelling
print('Correct rows:   ', (main_df['condition'] == 'Bipolar Disorder').sum())

# Look for ANY value containing "Bipolar", in case other variants are hiding
# regex=False means "search for this exact text", nothing fancy
bipolar_values = main_df[main_df['condition'].str.contains('Bipolar', regex=False)]['condition']

# Show each different spelling and how many rows it has
display(bipolar_values.value_counts().to_frame('row_count'))

Misspelled rows: 4224
Correct rows:    0


,row_count
condition,
Bipolar Disorde,4224


In [15]:
# Cell 2: apply the rename

# .replace() swaps the wrong text for the right text
# The dictionary format is {'wrong': 'right'}
# The result is saved back into the same column
main_df['condition'] = main_df['condition'].replace({'Bipolar Disorde': 'Bipolar Disorder'})

In [16]:
# Cell 3: AFTER check

# The misspelling should now be 0
print('Misspelled rows:', (main_df['condition'] == 'Bipolar Disorde').sum())

# The correct spelling should now be: old correct count + old misspelled count
print('Correct rows:   ', (main_df['condition'] == 'Bipolar Disorder').sum())

Misspelled rows: 0
Correct rows:    4224


In [17]:
# Cell 4: look for junk values (read-only, nothing is changed)

# True for rows where condition contains "</span>" (leftover HTML)
# regex=False means "search for this exact text"
has_span = main_df['condition'].str.contains('</span>', regex=False)

# How many rows have this problem?
print('Rows with </span>:', has_span.sum())

# Which values are they, and how many rows each? Show the top 10
display(main_df[has_span]['condition'].value_counts().head(10).to_frame('row_count'))

Rows with </span>: 900


,row_count
condition,
0</span> users found this comment helpful.,104
2</span> users found this comment helpful.,85
3</span> users found this comment helpful.,84
1</span> users found this comment helpful.,83
4</span> users found this comment helpful.,66
6</span> users found this comment helpful.,46
5</span> users found this comment helpful.,40
7</span> users found this comment helpful.,37
8</span> users found this comment helpful.,31


In [18]:
# Cell 5: look at the rarest condition values

# Count how many rows each condition has (biggest first)
cond_counts = main_df['condition'].value_counts()

# Show the 30 rarest conditions
# Junk values are usually rare, because each one appears only a few times
display(cond_counts.tail(30).to_frame('row_count'))

,row_count
condition,
"Tuberculosis, Active",1
Cerebrovascular Insufficiency,1
Dermatitis Herpeti,1
Anti NMDA Receptor Encephalitis,1
72</span> users found this comment helpful.,1
Gonadotropin Inhibition,1
unctional Gastric Disorde,1
Acetaminophen Overdose,1
"Paragonimus westermani, Lung Fluke",1


In [19]:
# Cell 6: find rows with HTML-looking characters

# True for rows where the condition contains < or >
# [<>] means "either of these two characters"
has_tag = main_df['condition'].str.contains('[<>]', regex=True)

# True for rows with </span> (same as Cell 4, repeated so this cell works alone)
has_span = main_df['condition'].str.contains('</span>', regex=False)

# Rows with < or > that are NOT the </span> group we already know about
other_tag = has_tag & ~has_span

print('Rows with < or > but not </span>:', other_tag.sum())

# Show which values they are (top 10)
display(main_df[other_tag]['condition'].value_counts().head(10).to_frame('row_count'))

Rows with < or > but not </span>: 0


,row_count
condition,


In [20]:
# Cell 7: add up all the junk we have found so far

# | means "or": True if the row is in either group
all_junk = has_span | has_tag

print('Total junk rows so far:', all_junk.sum())
print('Compare to the ~2,450 in your notes')

Total junk rows so far: 900
Compare to the ~2,450 in your notes


In [21]:
# Cell 8: look for values cut off at the end (read-only, nothing is changed)

# True for rows where the condition ends with "Disorde" (a cut-off "Disorder")
# .str.endswith() checks the end of the text
ends_disorde = main_df['condition'].str.endswith('Disorde')

# How many rows have this problem?
print('Rows ending in "Disorde":', ends_disorde.sum())

# Which values are they, and how many rows each?
display(main_df[ends_disorde]['condition'].value_counts().to_frame('row_count'))

Rows ending in "Disorde": 6674


,row_count
condition,
Major Depressive Disorde,1607
Panic Disorde,1463
Generalized Anxiety Disorde,1164
Obsessive Compulsive Disorde,579
Schizoaffective Disorde,396
Social Anxiety Disorde,389
Post Traumatic Stress Disorde,314
Premenstrual Dysphoric Disorde,298
Borderline Personality Disorde,151


In [22]:
# Cell 9: check whether the correct spellings also exist (read-only)

# True for rows ending with the FULL word "Disorder"
full_disorder = main_df['condition'].str.endswith('Disorder')
print('Rows ending in "Disorder":', full_disorder.sum())

# Show which conditions they are (top 10)
display(main_df[full_disorder]['condition'].value_counts().head(10).to_frame('row_count'))

# Does the full word "Fibromyalgia" exist next to the cut-off "ibromyalgia"?
print('Fibromyalgia:', (main_df['condition'] == 'Fibromyalgia').sum())
print('ibromyalgia: ', (main_df['condition'] == 'ibromyalgia').sum())

Rows ending in "Disorder": 4224


,row_count
condition,
Bipolar Disorder,4224


Fibromyalgia: 0
ibromyalgia:  1791


In [23]:
main_df.shape

(160398, 8)

In [24]:
# Set aside the junk rows

# True for rows where condition contains "</span>" (leftover HTML)
# Recalculated from main_df, because the old has_span was built before the split
mask_span = main_df['condition'].str.contains('</span>', regex=False)

# Junk rows go into their own table (.copy() makes it separate from main_df)
junk_cond_df = main_df[mask_span].copy()

# Keep everything else in main_df
# ~ flips True and False, so ~mask_span means "NOT junk"
# reset_index(drop=True) restarts the row numbers at 0
rows_before = len(main_df)
main_df = main_df[~mask_span].copy().reset_index(drop=True)

# Check that nothing got lost
print('Before:        ', rows_before)
print('main_df now:   ', len(main_df))
print('junk_cond_df:  ', len(junk_cond_df))

# Stops with an error if the rows don't add up
assert len(main_df) + len(junk_cond_df) == rows_before

# Save the set-aside rows for later sessions
junk_cond_df.to_csv('data/junk_condition_rows.csv', index=False)

Before:         160398
main_df now:    159498
junk_cond_df:   900


In [25]:
# Cell 10: BEFORE check

# True for rows where condition ends with the cut-off "Disorde"
# .str.endswith() checks the end of the text
ends_disorde = main_df['condition'].str.endswith('Disorde')

# How many rows have the cut-off ending?
print('Rows ending in "Disorde":', ends_disorde.sum())

# How many rows already end in the full "Disorder"? (this is Bipolar, 4,224)
print('Rows ending in "Disorder":', main_df['condition'].str.endswith('Disorder').sum())

Rows ending in "Disorde": 6674
Rows ending in "Disorder": 4224


In [26]:
# Cell 11: apply the fix

# .str.replace() swaps text inside each value
# 'Disorde$' means "Disorde at the very END of the text"
# The $ is why a value that already ends in "Disorder" is not touched
# regex=True tells pandas that the pattern uses special symbols like $
main_df['condition'] = main_df['condition'].str.replace('Disorde$', 'Disorder', regex=True)

In [27]:
# Cell 12: AFTER check

# The cut-off ending should now be 0
print('Rows ending in "Disorde":', main_df['condition'].str.endswith('Disorde').sum())

# The full ending should now be: old Disorder count + old Disorde count
print('Rows ending in "Disorder":', main_df['condition'].str.endswith('Disorder').sum())

Rows ending in "Disorde": 0
Rows ending in "Disorder": 10898


In [28]:
# Cell 13: list the cut-off-at-the-start values (read-only)

# Count rows per condition in the current main_df
cond_counts = main_df['condition'].value_counts()

# Keep only conditions whose first character is a lowercase letter
# r'^[a-z]' means "starts with a lowercase letter"
lowercase_start = cond_counts[cond_counts.index.str.match(r'^[a-z]')]

# Show how many different values there are and how many rows they cover
print('Different values:', len(lowercase_start))
print('Rows covered:    ', lowercase_start.sum())

# Show them all, biggest first
display(lowercase_start.to_frame('row_count'))

Different values: 37
Rows covered:     2453


,row_count
condition,
ibromyalgia,1791
mance Anxiety,187
moterol),79
emale Infertility,65
atigue,53
min),45
ge (amlodipine / valsartan),33
acial Wrinkles,32
moterol / mometasone),29


In [29]:
# Cell 14: BEFORE check

# Count how many rows start with a lowercase letter right now
# r'^[a-z]' means "the first character is a lowercase letter"
starts_lower = main_df['condition'].str.match(r'^[a-z]')
print('Rows starting with lowercase:', starts_lower.sum())

Rows starting with lowercase: 2453


In [30]:
# Cell 15: the lookup dictionary and the repair

# Format is {'cut-off value': 'correct value'}
# Only values that are clearly a condition go in here
fix_map = {
    # Your confirmed ones
    'ibromyalgia': 'Fibromyalgia',
    'mance Anxiety': 'Performance Anxiety',
    'emale Infertility': 'Female Infertility',
    'atigue': 'Fatigue',
    'acial Wrinkles': 'Facial Wrinkles',
    'zen Shoulde': 'Frozen Shoulder',   # cut off at BOTH ends
    # New ones from your screenshot: please check each one
    'lic Acid Deficiency': 'Folic Acid Deficiency',
    'amilial Mediterranean Feve': 'Familial Mediterranean Fever',  # both ends
    'cal Segmental Glomerulosclerosis': 'Focal Segmental Glomerulosclerosis',
    'actor IX Deficiency': 'Factor IX Deficiency',
    'ailure to Thrive': 'Failure to Thrive',
    'amilial Cold Autoinflammatory Syndrome': 'Familial Cold Autoinflammatory Syndrome',
    'ungal Pneumonia': 'Fungal Pneumonia',
    'ungal Infection Prophylaxis': 'Fungal Infection Prophylaxis',
    'ibrocystic Breast Disease': 'Fibrocystic Breast Disease',
    'llicular Lymphoma': 'Follicular Lymphoma',
    'm Pain Disorder': 'Somatoform Pain Disorder',
    'unctional Gastric Disorder': 'Functional Gastric Disorder',
    'acial Lipoatrophy': 'Facial Lipoatrophy',
}

# Swap every cut-off value for its correct version
# Values that are not in the dictionary are left alone
main_df['condition'] = main_df['condition'].replace(fix_map)

In [31]:
# Cell 16: AFTER check, and look at what is left (read-only)

# How many rows still start with a lowercase letter?
starts_lower = main_df['condition'].str.match(r'^[a-z]')
print('Rows still starting with lowercase:', starts_lower.sum())

# Show the leftover values, with each one's most common drug
# .agg() with 'count' gives the row count, and the lambda gives the top drug
leftover = main_df[starts_lower].groupby('condition').agg(
    n_rows=('drugName', 'count'),                          # how many rows
    top_drug=('drugName', lambda x: x.value_counts().index[0])  # most common drug
)
display(leftover.sort_values('n_rows', ascending=False))

Rows still starting with lowercase: 277


,n_rows,top_drug
condition,,
moterol),79,Budesonide / formoterol
min),45,Metformin
ge (amlodipine / valsartan),33,Amlodipine / valsartan
moterol / mometasone),29,Formoterol / mometasone
min / sitagliptin),26,Metformin / sitagliptin
eve,12,Acetaminophen
mulation) (phenylephrine),12,Phenylephrine
min / saxagliptin),10,Metformin / saxagliptin
von Willebrand's Disease,7,Stimate


In [32]:
# Set aside the leftover cut-off rows (drug fragments and unreadable values)

# True for rows whose condition starts with a lowercase letter
# r'^[a-z]' means "the first character is a lowercase letter"
starts_lower = main_df['condition'].str.match(r'^[a-z]')

# True for the one real condition we want to KEEP
is_von_w = main_df['condition'] == "von Willebrand's Disease"

# Rows to set aside: starts lowercase AND is not von Willebrand's
# & means "and", ~ flips True and False
mask_leftover = starts_lower & ~is_von_w

# Put those rows in their own table (.copy() makes it separate from main_df)
leftover_cond_df = main_df[mask_leftover].copy()

# Keep everything else in main_df and restart the row numbers at 0
rows_before = len(main_df)
main_df = main_df[~mask_leftover].copy().reset_index(drop=True)

# Check that nothing got lost
print('Before:           ', rows_before)
print('main_df now:      ', len(main_df))
print('leftover_cond_df: ', len(leftover_cond_df))

# Stops with an error if the rows don't add up
assert len(main_df) + len(leftover_cond_df) == rows_before

# Save the set-aside rows for later sessions
leftover_cond_df.to_csv('data/leftover_condition_rows.csv', index=False)

Before:            159498
main_df now:       159228
leftover_cond_df:  270


In [33]:
# Cell 17: count text artifacts in the review column (read-only, nothing is changed)

# A dictionary of {name: pattern to search for}
# regex=False in the loop below means "search for this exact text"
artifacts = {
    '&#039; (apostrophe code)': '&#039;',
    '&quot; (quote code)': '&quot;',
    '&amp; (ampersand code)': '&amp;',
    '<br> (line break tag)': '<br',
    'Newline character (\\n)': '\n',
    'Carriage return (\\r)': '\r',
}

# Start an empty list to collect the results
results = []

# Go through each artifact one at a time
for name, pattern in artifacts.items():
    # True for reviews that contain this text, .sum() counts the Trues
    count = main_df['review'].str.contains(pattern, regex=False).sum()
    # Save the name and the count
    results.append({'artifact': name, 'n_reviews': count})

# Turn the list into a dataframe and show it
display(pd.DataFrame(results))

,artifact,n_reviews
0,&#039; (apostrophe code),99360
1,&quot; (quote code),9454
2,&amp; (ampersand code),6390
3,<br> (line break tag),0
4,Newline character (\n),17815
5,Carriage return (\r),17815


In [ ]:
# Cell 18: look at a few real examples (read-only)

# Show 5 random reviews that contain the apostrophe code
# .sample(5, random_state=1) picks 5 random rows; random_state makes it repeatable
# If this cell shows an error, the count in Cell 17 was probably 0, so skip it
has_apos = main_df['review'].str.contains('&#039;', regex=False)
for text in main_df[has_apos]['review'].sample(5, random_state=1):
    print(text[:300])   # print only the first 300 characters
    print('-----')

In [35]:
# Cell 19: confirm that \r and \n come as a pair (read-only, nothing is changed)

# True for reviews that contain the pair \r\n, one right after the other
has_pair = main_df['review'].str.contains('\r\n', regex=False)

# True for reviews that contain a \n anywhere
has_n = main_df['review'].str.contains('\n', regex=False)

print('Reviews with \\r\\n pair:', has_pair.sum())
print('Reviews with any \\n:   ', has_n.sum())

# Reviews that have \n but NOT as part of a pair (should be 0 if all are pairs)
print('Lone \\n (no \\r before it):', (has_n & ~has_pair).sum())

Reviews with \r\n pair: 17815
Reviews with any \n:    17815
Lone \n (no \r before it): 0


In [36]:
# Cell 19: BEFORE check (read-only)

# Count reviews that contain each artifact right now
# regex=False means "search for this exact text"
print('Reviews with &#039;:', main_df['review'].str.contains('&#039;', regex=False).sum())
print('Reviews with &quot;:', main_df['review'].str.contains('&quot;', regex=False).sum())
print('Reviews with &amp; :', main_df['review'].str.contains('&amp;', regex=False).sum())
print('Reviews with \\r    :', main_df['review'].str.contains('\r', regex=False).sum())
print('Reviews with \\n    :', main_df['review'].str.contains('\n', regex=False).sum())

Reviews with &#039;: 99360
Reviews with &quot;: 9454
Reviews with &amp; : 6390
Reviews with \r    : 17815
Reviews with \n    : 17815


In [37]:
# Cell 20: apply the cleaning

# html is a built-in Python module, so there is nothing to install
import html

# Step 1: turn HTML codes into real characters
# .apply() runs html.unescape on each review, one at a time
# For example: "I&#039;m" becomes "I'm"
main_df['review_clean'] = main_df['review'].apply(html.unescape)

# Step 2: turn every run of whitespace into a single space
# \s+ means "one or more whitespace characters" (spaces, \r, \n, tabs)
# This removes the line breaks and the extra spaces in one go
main_df['review_clean'] = main_df['review_clean'].str.replace(r'\s+', ' ', regex=True)

# Step 3: remove spaces at the very start and end of each review
main_df['review_clean'] = main_df['review_clean'].str.strip()

In [38]:
# Cell 21: AFTER check (read-only)

# The row count should not have changed
print('Rows in main_df:', len(main_df))

# All of these should now be 0
print('Reviews with &#039;:', main_df['review_clean'].str.contains('&#039;', regex=False).sum())
print('Reviews with &quot;:', main_df['review_clean'].str.contains('&quot;', regex=False).sum())
print('Reviews with &amp; :', main_df['review_clean'].str.contains('&amp;', regex=False).sum())
print('Reviews with \\r    :', main_df['review_clean'].str.contains('\r', regex=False).sum())
print('Reviews with \\n    :', main_df['review_clean'].str.contains('\n', regex=False).sum())

# Reviews that still contain two spaces in a row (should be 0)
print('Reviews with double space:', main_df['review_clean'].str.contains('  ', regex=False).sum())

# Look at 3 reviews side by side: original vs cleaned (first 200 characters)
for i in main_df[main_df['review'].str.contains('&#039;', regex=False)].sample(3, random_state=1).index:
    print('BEFORE:', main_df.loc[i, 'review'][:200])
    print('AFTER: ', main_df.loc[i, 'review_clean'][:200])
    print('-----')

Rows in main_df: 159228
Reviews with &#039;: 0
Reviews with &quot;: 0
Reviews with &amp; : 0
Reviews with \r    : 0
Reviews with \n    : 0
Reviews with double space: 0


In [39]:
# Cell 22: count duplicates (read-only, nothing is changed)

# Columns that define "the same review"
# review_clean is used so copies that differ only in HTML codes still match
key_cols = ['review_clean', 'condition', 'date']

# True for every row that has at least one identical twin (keep=False marks ALL copies)
dup_all = main_df.duplicated(subset=key_cols, keep=False)

# True only for the EXTRA copies (the first copy is not marked)
dup_extra = main_df.duplicated(subset=key_cols, keep='first')

print('Rows involved in duplicates (all copies):', dup_all.sum())
print('Extra copies that would be removed:      ', dup_extra.sum())

Rows involved in duplicates (all copies): 95642
Extra copies that would be removed:       47836


In [ ]:
# Cell 23: look at a few duplicate groups (read-only)

# Keep only the duplicated rows and sort so copies sit next to each other
dups = main_df[dup_all].sort_values(key_cols)

# Show some columns of the first 12 rows
# review_clean is cut to 60 characters so the table stays readable
view = dups[['Unnamed: 0', 'drugName', 'condition', 'rating', 'date']].copy()
view['review_start'] = dups['review_clean'].str[:60]
display(view.head(12))

In [41]:
# Cell 24: the same review under DIFFERENT conditions (read-only)

# For each review text, count how many different conditions it appears under
conds_per_review = main_df.groupby('review_clean')['condition'].nunique()

# Reviews that appear under 2 or more conditions
multi = conds_per_review[conds_per_review > 1]
print('Review texts under 2+ different conditions:', len(multi))

Review texts under 2+ different conditions: 130


In [42]:
# Cell 25: are the duplicate groups really the same review? (read-only)

# For each group of identical reviews, count rows, different ratings and different drug names
# groupby(key_cols) makes one row per group; .agg() calculates the three numbers
grp = main_df[dup_all].groupby(key_cols).agg(
    n_rows=('rating', 'count'),         # how many copies in the group
    n_ratings=('rating', 'nunique'),    # how many different ratings
    n_drugs=('drugName', 'nunique'),    # how many different drug names
)

print('Number of duplicate groups:', len(grp))

# How many groups have 2 copies, 3 copies, and so on?
display(grp['n_rows'].value_counts().to_frame('n_groups'))

# Groups where the copies have different ratings (should be close to 0 if they are true duplicates)
print('Groups with different ratings:', (grp['n_ratings'] > 1).sum())

# Groups where all copies have the SAME drug name (a different kind of duplicate)
print('Groups with only one drug name:', (grp['n_drugs'] == 1).sum())

Number of duplicate groups: 47806


,n_groups
n_rows,
2,47785
3,14
4,6
6,1


Groups with different ratings: 2
Groups with only one drug name: 8


In [ ]:
# Cell 26: look at the 130 review texts under 2+ conditions (read-only)

# Keep only rows whose review text is in the "multi" list from Cell 24
multi_df = main_df[main_df['review_clean'].isin(multi.index)].copy()

# Add a short preview of the text and a word count
multi_df['review_start'] = multi_df['review_clean'].str[:50]
multi_df['n_words'] = multi_df['review_clean'].str.split().str.len()

# Sort so identical texts sit together, then show the first 20 rows
cols = ['drugName', 'condition', 'rating', 'date', 'n_words', 'review_start']
display(multi_df.sort_values('review_clean')[cols].head(20))

In [44]:
# Read-only: how many review texts sit under BOTH Weight Loss and Obesity?

# Keep only the rows with these two conditions
wl_ob = main_df[main_df['condition'].isin(['Weight Loss', 'Obesity'])]

# For each review text, count how many different conditions it appears under
n_conds = wl_ob.groupby('review_clean')['condition'].nunique()

# Texts that appear under both (nunique == 2)
print('Review texts under both Weight Loss and Obesity:', (n_conds == 2).sum())

Review texts under both Weight Loss and Obesity: 0


In [45]:
# Cell 27: BEFORE check (read-only)

# Count how many rows each of these conditions has right now
print('Weight Loss:      ', (main_df['condition'] == 'Weight Loss').sum())
print('Obesity:          ', (main_df['condition'] == 'Obesity').sum())
print('Weight Management:', (main_df['condition'] == 'Weight Management').sum())  # should be 0

Weight Loss:       3609
Obesity:           3568
Weight Management: 0


In [46]:
# Cell 28: apply the merge

# Swap both old names for the new combined name
# The dictionary format is {'old': 'new'}
main_df['condition'] = main_df['condition'].replace({
    'Weight Loss': 'Weight Management',
    'Obesity': 'Weight Management',
})

In [47]:
# Cell 29: AFTER check (read-only)

# The old names should now be 0
print('Weight Loss:      ', (main_df['condition'] == 'Weight Loss').sum())
print('Obesity:          ', (main_df['condition'] == 'Obesity').sum())

# The new name should equal Weight Loss + Obesity from Cell 27
print('Weight Management:', (main_df['condition'] == 'Weight Management').sum())

# The row count should not have changed
print('Rows in main_df:  ', len(main_df))

Weight Loss:       0
Obesity:           0
Weight Management: 7177
Rows in main_df:   159228


In [48]:
# Cell 30: look at the biggest conditions (read-only)

# Count rows per condition, biggest first
cond_counts = main_df['condition'].value_counts()

# Show the top 40 conditions
# Cut-off endings will look like words that stop mid-way
display(cond_counts.head(40).to_frame('row_count'))

,row_count
condition,
Birth Control,28788
Depression,9069
Weight Management,7177
Pain,6145
Anxiety,5904
Acne,5588
Bipolar Disorder,4224
Insomnia,3673
ADHD,3383


In [49]:
# Cell 31: how many conditions are there, and how many rows do the small ones hold? (read-only)

print('Different conditions:', len(cond_counts))

# Rows held by conditions with fewer than 100 rows
small = cond_counts[cond_counts < 100]
print('Conditions with < 100 rows:', len(small))
print('Rows they cover:           ', small.sum())

Different conditions: 791
Conditions with < 100 rows: 643
Rows they cover:            10910


In [50]:
# Cell 32: is the full spelling of "Overactive Bladder" present anywhere? (read-only)

# Count rows for the cut-off spelling and the full spelling
print('Overactive Bladde: ', (main_df['condition'] == 'Overactive Bladde').sum())
print('Overactive Bladder:', (main_df['condition'] == 'Overactive Bladder').sum())

Overactive Bladde:  683
Overactive Bladder: 0


In [51]:
# Cell 33: find other possible cut-offs (read-only, nothing is changed)

# Count rows per condition
cond_counts = main_df['condition'].value_counts()

# Make a plain list of all condition names
names = list(cond_counts.index)

# An empty list to collect what we find
rows = []

# Compare every name (short) with every other name (full)
for short in names:
    for full in names:
        # How many more characters does "full" have than "short"?
        extra = len(full) - len(short)
        # Keep the pair if full is longer by 1 to 6 characters
        # AND full starts with exactly the text of short
        if 0 < extra <= 6 and full.startswith(short):
            rows.append({
                'possible_cutoff': short,
                'rows_cutoff': cond_counts[short],
                'possible_full': full,
                'rows_full': cond_counts[full],
            })

# Show the pairs, biggest first
display(pd.DataFrame(rows).sort_values('rows_cutoff', ascending=False))

,possible_cutoff,rows_cutoff,possible_full,rows_full
0,Pain,6145,Pain/Feve,13
1,Glioblastoma Multi,6,Glioblastoma Multiforme,5
2,Dermatitis Herpeti,1,Dermatitis Herpetiformis,1


In [52]:
display(cond_counts.iloc[40:148].to_frame('row_count'))

,row_count
condition,
Hyperhidrosis,677
Hepatitis C,656
Back Pain,616
HIV Infection,615
High Cholesterol,611
...,...
Dietary Supplementation,102
Human Papilloma Virus,102
"Breast Cancer, Metastatic",102


In [53]:
# Cell 34: show conditions ranked 41 to 80 (read-only)

# option_context changes the display setting for this cell only
# 'display.max_rows', None means "show every row, don't hide the middle"
with pd.option_context('display.max_rows', None):
    display(cond_counts.iloc[40:80].to_frame('row_count'))

,row_count
condition,
Hyperhidrosis,677
Hepatitis C,656
Back Pain,616
HIV Infection,615
High Cholesterol,611
Psoriasis,598
Schizophrenia,594
"Constipation, Chronic",591
Obsessive Compulsive Disorder,579


In [54]:
# Cell 35: show conditions ranked 81 to 120 (read-only)

with pd.option_context('display.max_rows', None):
    display(cond_counts.iloc[80:120].to_frame('row_count'))

,row_count
condition,
Inflammatory Conditions,303
Premenstrual Dysphoric Disorder,298
Nasal Congestion,290
Pneumonia,276
Chlamydia Infection,273
Breast Cance,272
Atrial Fibrillation,272
Urticaria,264
Helicobacter Pylori Infection,260


In [55]:
# Cell 36: show conditions ranked 121 to 148 (read-only)

with pd.option_context('display.max_rows', None):
    display(cond_counts.iloc[120:148].to_frame('row_count'))

,row_count
condition,
COPD,159
Ovarian Cysts,152
Otitis Media,152
Borderline Personality Disorder,151
"Hypothyroidism, After Thyroid Removal",151
Dry Eye Disease,147
Bacterial Skin Infection,146
Diverticulitis,142
"Diabetes, Type 1",140


In [56]:
# Cell 37
# The lookup: {'cut-off value': 'correct value'}
# Only values you confirmed as cut-off conditions go in here
fix_map2 = {
    'Overactive Bladde': 'Overactive Bladder',
    'Not Listed / Othe': 'Not Listed / Other',
    'Breast Cance': 'Breast Cancer',
    'Prostate Cance': 'Prostate Cancer',      # new: you spotted this one
    'Vertig': 'Vertigo',                      # new: you spotted this one
    'Glioblastoma Multi': 'Glioblastoma Multiforme',
    'Dermatitis Herpeti': 'Dermatitis Herpetiformis',
    'Stomach Ulce': 'Stomach Ulcer',
    'Pain/Feve': 'Pain/Fever',
}


# Count rows for each cut-off value and for its correct version
for wrong, right in fix_map2.items():
    n_wrong = (main_df['condition'] == wrong).sum()
    n_right = (main_df['condition'] == right).sum()
    print(f'{wrong}: {n_wrong}  ->  {right}: {n_right}')

# Total rows that will be changed
print('Rows to change:', main_df['condition'].isin(fix_map2.keys()).sum())

Overactive Bladde: 683  ->  Overactive Bladder: 0
Not Listed / Othe: 439  ->  Not Listed / Other: 0
Breast Cance: 272  ->  Breast Cancer: 0
Prostate Cance: 123  ->  Prostate Cancer: 0
Vertig: 101  ->  Vertigo: 0
Glioblastoma Multi: 6  ->  Glioblastoma Multiforme: 5
Dermatitis Herpeti: 1  ->  Dermatitis Herpetiformis: 1
Stomach Ulce: 72  ->  Stomach Ulcer: 0
Pain/Feve: 13  ->  Pain/Fever: 0
Rows to change: 1710


In [57]:
# Cell 38: apply the repair

# .replace() swaps each cut-off value for its correct version
# Values that are not in the dictionary are left alone
main_df['condition'] = main_df['condition'].replace(fix_map2)

In [58]:
# Cell 39: AFTER check (read-only)

# Every cut-off value should now be 0
for wrong, right in fix_map2.items():
    n_wrong = (main_df['condition'] == wrong).sum()
    n_right = (main_df['condition'] == right).sum()
    print(f'{wrong}: {n_wrong}  ->  {right}: {n_right}')

# The row count should not have changed
print('Rows in main_df:', len(main_df))

Overactive Bladde: 0  ->  Overactive Bladder: 683
Not Listed / Othe: 0  ->  Not Listed / Other: 439
Breast Cance: 0  ->  Breast Cancer: 272
Prostate Cance: 0  ->  Prostate Cancer: 123
Vertig: 0  ->  Vertigo: 101
Glioblastoma Multi: 0  ->  Glioblastoma Multiforme: 11
Dermatitis Herpeti: 0  ->  Dermatitis Herpetiformis: 2
Stomach Ulce: 0  ->  Stomach Ulcer: 72
Pain/Feve: 0  ->  Pain/Fever: 13
Rows in main_df: 159228


In [59]:
# Cell 40: count duplicates again (read-only, nothing is changed)

# Same key as before: the three columns that define "the same review"
key_cols = ['review_clean', 'condition', 'date']

# True only for the EXTRA copies (the first copy is not marked)
dup_extra = main_df.duplicated(subset=key_cols, keep='first')

# Before the repairs this was 47,836
print('Extra copies that would be removed:', dup_extra.sum())

Extra copies that would be removed: 47842


In [60]:
# Cell 41: remove the extra copies and save them (run ONCE)

# Mark the extra copies (the first copy of each review is kept)
mask_dup = main_df.duplicated(subset=key_cols, keep='first')

# The removed rows go into their own table
dup_removed_df = main_df[mask_dup].copy()

# Keep everything else, and restart the row numbers at 0
rows_before = len(main_df)
main_df = main_df[~mask_dup].copy().reset_index(drop=True)

# Check that nothing got lost
print('Before:          ', rows_before)
print('main_df now:     ', len(main_df))
print('dup_removed_df:  ', len(dup_removed_df))
assert len(main_df) + len(dup_removed_df) == rows_before

# Save the removed rows for later sessions
dup_removed_df.to_csv('data/duplicate_rows_removed.csv', index=False)

Before:           159228
main_df now:      111386
dup_removed_df:   47842


In [61]:
# Cell 42: re-count review texts under 2+ conditions (read-only)

# For each review text, count how many different conditions it appears under
conds_per_review = main_df.groupby('review_clean')['condition'].nunique()

# Keep only texts that appear under 2 or more conditions
multi = conds_per_review[conds_per_review > 1]
print('Review texts under 2+ conditions:', len(multi))

Review texts under 2+ conditions: 124


In [62]:
# Cell 43: sort them into short and long texts (read-only)

# Rows whose review text is in the "multi" list
multi_df = main_df[main_df['review_clean'].isin(multi.index)].copy()

# Word count of each review
multi_df['n_words'] = multi_df['review_clean'].str.split().str.len()

# Short texts (under 10 words) are probably different people writing "Awesome."
# Long texts are more likely the SAME review filed under two conditions
print('Rows in these groups:     ', len(multi_df))
print('Rows with under 10 words: ', (multi_df['n_words'] < 10).sum())
print('Rows with 10+ words:      ', (multi_df['n_words'] >= 10).sum())

# Show the long ones, sorted so identical texts sit together
cols = ['drugName', 'condition', 'rating', 'date', 'n_words']
long_df = multi_df[multi_df['n_words'] >= 10].sort_values('review_clean')
display(long_df[cols].head(30))

Rows in these groups:      374
Rows with under 10 words:  313
Rows with 10+ words:       61


,drugName,condition,rating,date,n_words
73954,Amphetamine / dextroamphetamine,Narcolepsy,9,3-Jul-13,28
33521,Adderall,ADHD,9,3-Jul-13,28
105055,Venlafaxine,Anxiety,10,25-Jul-16,74
96270,Venlafaxine,Depression,9,22-Jul-16,74
51774,Tadalafil,Not Listed / Other,10,25-Jun-08,25
31265,Cialis,Erectile Dysfunction,10,25-Jun-08,25
102885,Sucralfate,Not Listed / Other,8,27-Oct-08,42
9693,Sucralfate,Stomach Ulcer,8,27-Oct-08,42
96920,Propranolol,Pe,10,15-Mar-13,31
9820,Inderal,Performance Anxiety,10,15-Mar-13,31


In [63]:
# Cell 44: BEFORE check (read-only)

# 'Pe' must match the whole value exactly, so longer names that start with Pe are not counted
print('Pe:                 ', (main_df['condition'] == 'Pe').sum())
print('Performance Anxiety:', (main_df['condition'] == 'Performance Anxiety').sum())

Pe:                  5
Performance Anxiety: 194


In [64]:
# Cell 45: apply the repair, then recheck (run ONCE)

# Swap the cut-off value for the correct one
main_df['condition'] = main_df['condition'].replace({'Pe': 'Performance Anxiety'})

# AFTER check: Pe should be 0, and Performance Anxiety should be the old two counts added together
print('Pe:                 ', (main_df['condition'] == 'Pe').sum())
print('Performance Anxiety:', (main_df['condition'] == 'Performance Anxiety').sum())

# Are there new duplicate copies now? (read-only count)
key_cols = ['review_clean', 'condition', 'date']
print('New extra copies:   ', main_df.duplicated(subset=key_cols, keep='first').sum())

Pe:                  0
Performance Anxiety: 199
New extra copies:    4


In [65]:
# Cell 46: remove the new duplicate copies (run ONCE)

# Mark the extra copies
mask_dup2 = main_df.duplicated(subset=key_cols, keep='first')

# Put them in a table and add them to the earlier removed duplicates
new_dups = main_df[mask_dup2].copy()
dup_removed_df = pd.concat([dup_removed_df, new_dups], ignore_index=True)

# Keep everything else in main_df
rows_before = len(main_df)
main_df = main_df[~mask_dup2].copy().reset_index(drop=True)

# Check that nothing got lost
print('Before:       ', rows_before)
print('main_df now:  ', len(main_df))
print('Newly removed:', len(new_dups))
assert len(main_df) + len(new_dups) == rows_before

# Save the full removed-duplicates table again (it now includes the new ones)
dup_removed_df.to_csv('data/duplicate_rows_removed.csv', index=False)

Before:        111386
main_df now:   111382
Newly removed: 4


In [66]:
# Cell 47: set aside long review texts that sit under 2+ conditions (run ONCE)

# Rebuild: for each review text, how many different conditions does it appear under?
conds_per_review = main_df.groupby('review_clean')['condition'].nunique()
multi = conds_per_review[conds_per_review > 1]

# True for rows whose text is in that list
in_multi = main_df['review_clean'].isin(multi.index)

# Word count of every review
n_words = main_df['review_clean'].str.split().str.len()

# Rows to set aside: in a multi-condition group AND 10+ words
# (short ones like "Awesome." are different people, so they stay)
mask_long = in_multi & (n_words >= 10)

# Put them in their own table
multi_label_df = main_df[mask_long].copy()

# Keep everything else in main_df
rows_before = len(main_df)
main_df = main_df[~mask_long].copy().reset_index(drop=True)

# Check that nothing got lost
print('Before:          ', rows_before)
print('main_df now:     ', len(main_df))
print('multi_label_df:  ', len(multi_label_df))
assert len(main_df) + len(multi_label_df) == rows_before

# Save the set-aside rows
multi_label_df.to_csv('data/multi_condition_rows.csv', index=False)

Before:           111382
main_df now:      111327
multi_label_df:   55


In [67]:
# Cell 48: how the counts fall off (read-only)

# Count rows per condition, biggest first
cond_counts = main_df['condition'].value_counts()

# How many different conditions are there now?
print('Different conditions:', len(cond_counts))

# For several possible top-N sizes, show how many rows are covered
# and what the smallest class in that set would have
for n in [10, 15, 20, 25, 30, 40, 50]:
    top = cond_counts.head(n)
    share = top.sum() / cond_counts.sum() * 100
    print(f'Top {n:>2}: covers {top.sum():>6} rows ({share:.1f}%), smallest class = {top.index[-1]} ({top.iloc[-1]} rows)')

Different conditions: 786
Top 10: covers  52175 rows (46.9%), smallest class = Vaginal Yeast Infection (2015 rows)
Top 15: covers  59863 rows (53.8%), smallest class = Fibromyalgia (1256 rows)
Top 20: covers  65804 rows (59.1%), smallest class = Constipation (1121 rows)
Top 25: covers  71021 rows (63.8%), smallest class = Urinary Tract Infection (954 rows)
Top 30: covers  75172 rows (67.5%), smallest class = Erectile Dysfunction (725 rows)
Top 40: covers  81297 rows (73.0%), smallest class = Overactive Bladder (465 rows)
Top 50: covers  85619 rows (76.9%), smallest class = Psoriasis (415 rows)


In [68]:
# Cell 49: look at the top 50 (read-only)

# .to_frame() makes a table; display() shows it
display(cond_counts.head(50).to_frame('row_count'))

,row_count
condition,
Birth Control,18133
Depression,6287
Weight Management,5086
Pain,4453
Anxiety,4339
Acne,3955
Bipolar Disorder,2883
Insomnia,2833
ADHD,2191


In [69]:
# Cell 50: do any review texts sit under BOTH labels of a pair? (read-only)

# The two pairs you chose
pairs = [
    ('Depression', 'Major Depressive Disorder'),
    ('Anxiety', 'Anxiety and Stress'),
]

for a, b in pairs:
    # Keep only the rows with these two conditions
    sub = main_df[main_df['condition'].isin([a, b])]
    # For each review text, count how many different conditions it appears under
    n_conds = sub.groupby('review_clean')['condition'].nunique()
    # Texts that appear under both (nunique == 2)
    print(f'{a} + {b}: {(n_conds == 2).sum()} review texts under both')

Depression + Major Depressive Disorder: 0 review texts under both
Anxiety + Anxiety and Stress: 2 review texts under both


In [70]:
# Cell 51: keep only the top 30 conditions (run ONCE)

# Recount from the current main_df (the old cond_counts is a snapshot)
cond_counts = main_df['condition'].value_counts()

# The names of the 30 biggest conditions
top30 = cond_counts.head(30).index

# True for rows whose condition is NOT in the top 30
mask_out = ~main_df['condition'].isin(top30)

# Rows outside the class set go in their own table
out_of_set_df = main_df[mask_out].copy()

# Keep the rest and restart the row numbers at 0
rows_before = len(main_df)
main_df = main_df[~mask_out].copy().reset_index(drop=True)

# Check that nothing got lost
print('Before:          ', rows_before)
print('main_df now:     ', len(main_df))
print('out_of_set_df:   ', len(out_of_set_df))
print('Classes in main_df:', main_df['condition'].nunique())
assert len(main_df) + len(out_of_set_df) == rows_before

# Save the set-aside rows for later sessions
out_of_set_df.to_csv('data/out_of_class_set_rows.csv', index=False)

Before:           111327
main_df now:      75172
out_of_set_df:    36155
Classes in main_df: 30


In [71]:
# Cell 52: word-count distribution of review_clean (read-only)

# Count the words in each cleaned review
# .str.split() cuts the text at spaces, .str.len() counts the pieces
n_words = main_df['review_clean'].str.split().str.len()

# Summary numbers: min, quartiles, max
display(n_words.describe().to_frame('n_words'))

# How many reviews have 1, 2, 3 ... 10 words?
display(n_words[n_words <= 10].value_counts().sort_index().to_frame('n_reviews'))

,n_words
count,75172.000000
mean,89.137565
std,44.831465
min,1.000000
25%,54.000000
50%,91.000000
75%,129.000000
max,1894.000000


,n_reviews
review_clean,
1,99
2,140
3,178
4,206
5,213
6,222
7,221
8,228
9,249


In [72]:
# Cell 53: cumulative view (read-only)

# For several possible cutoffs, how many reviews would have FEWER words than the cutoff?
for cutoff in [3, 5, 8, 10, 15, 20]:
    n = (n_words < cutoff).sum()
    share = n / len(n_words) * 100
    print(f'Under {cutoff:>2} words: {n:>5} reviews ({share:.2f}%)')

Under  3 words:   239 reviews (0.32%)
Under  5 words:   623 reviews (0.83%)
Under  8 words:  1279 reviews (1.70%)
Under 10 words:  1756 reviews (2.34%)
Under 15 words:  3142 reviews (4.18%)
Under 20 words:  4604 reviews (6.12%)


In [ ]:
# Cell 54: read some actual short reviews (read-only)

# Show 15 random reviews with 5 words or fewer, with their condition
short_df = main_df[n_words <= 5]
display(short_df[['condition', 'rating', 'review_clean']].sample(15, random_state=1))

In [74]:
# Cell 55: set aside very short reviews (run ONCE)

# Count the words in each cleaned review (recalculated, so it matches the current main_df)
n_words = main_df['review_clean'].str.split().str.len()

# True for reviews with fewer than 5 words (so 1 to 4 words)
mask_short = n_words < 5

# Short reviews go in their own table (.copy() makes it separate from main_df)
short_reviews_df = main_df[mask_short].copy()

# Keep the rest and restart the row numbers at 0
rows_before = len(main_df)
main_df = main_df[~mask_short].copy().reset_index(drop=True)

# Check that nothing got lost
print('Before:            ', rows_before)
print('main_df now:       ', len(main_df))
print('short_reviews_df:  ', len(short_reviews_df))
assert len(main_df) + len(short_reviews_df) == rows_before

# Save the set-aside rows for later sessions
short_reviews_df.to_csv('data/short_review_rows.csv', index=False)

Before:             75172
main_df now:        74549
short_reviews_df:   623


In [75]:
# Save the cleaned working table for the preprocessing notebook
# index=False stops pandas from writing an extra column of row numbers
main_df.to_csv('data/cleaned_train.csv', index=False)

# Quick check: load it back and compare the shape
check = pd.read_csv('data/cleaned_train.csv')
print('Saved shape:', check.shape)
print('Classes:    ', check['condition'].nunique())

Saved shape: (74549, 9)
Classes:     30


In [76]:
# Cell 56: final shape and class counts table

# Shape = (rows, columns)
print('Final shape:', main_df.shape)

# Count rows per class, biggest first
# .to_frame() makes a table, and 'row_count' names the column
class_table = main_df['condition'].value_counts().to_frame('row_count')

# Add each class's share of the data as a percentage
# round(2) keeps two decimals
class_table['share_%'] = (class_table['row_count'] / len(main_df) * 100).round(2)

# Show every row (30 classes) instead of hiding the middle
with pd.option_context('display.max_rows', None):
    display(class_table)

# Save the table so it can go into the README
class_table.to_csv('data/final_class_counts.csv')

Final shape: (74549, 9)


,row_count,share_%
condition,,
Birth Control,18112,24.30
Depression,6224,8.35
Weight Management,5066,6.80
Pain,4356,5.84
Anxiety,4269,5.73
Acne,3943,5.29
Bipolar Disorder,2855,3.83
Insomnia,2797,3.75
ADHD,2163,2.90


In [77]:
# Cell 57: row-count log (typed in from the outputs above)

# Each entry: (step name, rows before, rows after, rows removed)
log_rows = [
    ('1. Missing condition set aside',         161297, 160398, 899),
    ('2. </span> junk set aside',              160398, 159498, 900),
    ('3. Leftover cut-off values set aside',   159498, 159228, 270),
    ('4. Duplicates removed',                  159228, 111386, 47842),
    ('5. New duplicates after Pe repair',      111386, 111382, 4),
    ('6. Multi-condition texts set aside',     111382, 111327, 55),
    ('7. Outside top 30 classes set aside',    111327, 75172, 36155),
    ('8. Reviews under 5 words set aside',     75172, 74549, 623),
]

# Turn the list into a table
log = pd.DataFrame(log_rows, columns=['step', 'rows_before', 'rows_after', 'rows_removed'])

# Check each line: before minus removed must equal after
assert (log['rows_before'] - log['rows_removed'] == log['rows_after']).all()

# Check the chain: each step must start where the last one ended
assert (log['rows_before'].iloc[1:].values == log['rows_after'].iloc[:-1].values).all()

# Check the total: everything removed plus what is left equals the original 161,297
assert log['rows_removed'].sum() + log['rows_after'].iloc[-1] == 161297

display(log)

# Save the log for the README
log.to_csv('data/cleaning_row_log.csv', index=False)

,step,rows_before,rows_after,rows_removed
0,1. Missing condition set aside,161297,160398,899
1,2. </span> junk set aside,160398,159498,900
2,3. Leftover cut-off values set aside,159498,159228,270
3,4. Duplicates removed,159228,111386,47842
4,5. New duplicates after Pe repair,111386,111382,4
5,6. Multi-condition texts set aside,111382,111327,55
6,7. Outside top 30 classes set aside,111327,75172,36155
7,8. Reviews under 5 words set aside,75172,74549,623


In [78]:
list(main_df['condition'].unique())

['ADHD',
 'Birth Control',
 'Emergency Contraception',
 'Bipolar Disorder',
 'Migraine Prevention',
 'Depression',
 'Weight Management',
 'Urinary Tract Infection',
 'Fibromyalgia',
 'Insomnia',
 'Vaginal Yeast Infection',
 'Panic Disorder',
 'Migraine',
 'Pain',
 'Osteoarthritis',
 'Constipation',
 'Bowel Preparation',
 'Muscle Spasm',
 'Diabetes, Type 2',
 'Smoking Cessation',
 'Anxiety',
 'Acne',
 'Erectile Dysfunction',
 'Chronic Pain',
 'Major Depressive Disorder',
 'Anxiety and Stress',
 'High Blood Pressure',
 'Abnormal Uterine Bleeding',
 'Generalized Anxiety Disorder',
 'Bacterial Infection']